In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())

Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [ ]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


Task 1: Heatmap: content by rating and release decade

In [ ]:
# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Filter to top ratings only
top_ratings = df['rating'].value_counts().nlargest(6).index.tolist()
filtered = df.loc[df['rating'].isin(top_ratings)].copy()

# Group and pivot
rating_decade = (
    filtered
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

pivot = rating_decade.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0)

# Create heatmap
fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    title='Netflix Titles by Rating and Release Decade'
)

# Layout improvements
fig.update_layout(
    xaxis_title='Release Decade',
    yaxis_title='Content Rating',
    height=500,
    width=800
)

fig.show()

Task 2 — Waterfall: Movie vs TV Show additions by year

In [ ]:
# Filter to Movies only
movies = df.loc[df['type'] == 'Movie'].copy()

# Group by added year
adds = (
    movies
    .groupby('added_year')
    .size()
    .reset_index(name='new_titles')
)

# Filter years
adds = adds.loc[
    (adds['added_year'] >= 2015) &
    (adds['added_year'] <= 2022)
].copy()

# Cumulative total
cumulative_total = adds['new_titles'].sum()

# Create waterfall chart
fig = go.Figure(
    go.Waterfall(
        x=[str(year) for year in adds['added_year']] + ['Total'],
        y=adds['new_titles'].tolist() + [cumulative_total],
        measure=['relative'] * len(adds) + ['total'],
        text=adds['new_titles'].tolist() + [cumulative_total],
        textposition='outside',
        increasing={"marker": {"color": "green"}},
        decreasing={"marker": {"color": "red"}},
        totals={"marker": {"color": "blue"}}
    )
)

# Layout improvements
fig.update_layout(
    title='Netflix Movie Additions by Year (2015–2022)',
    xaxis_title='Year',
    yaxis_title='Number of Titles Added',
    height=500,
    width=900,
    showlegend=False
)

fig.show()